## Część 1: SQLite + NumPy

In [1]:
import sqlite3
import json
import numpy as np

# 1. Inicjalizacja bazy SQLite i tabeli Movies
conn = sqlite3.connect("movies.db")
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS Movies")
cursor.execute("""
CREATE TABLE Movies (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT UNIQUE,
    embedding TEXT -- Wektor zapisany jako JSON array
)
""")
conn.commit()

# 2. Wstawienie filmów i ich embeddingów do bazy
filmy_db = {
    "Incepcja": [0.8, 0.3, 0.9],
    "Matrix": [0.75, 0.35, 0.85],
    "Toy Story": [0.2, 0.9, 0.1],
    "Shrek": [0.25, 0.85, 0.15],
    "Szeregowiec Ryan": [0.6, 0.1, 0.7],
}

for title, vec in filmy_db.items():
    cursor.execute(
        "INSERT INTO Movies (title, embedding) VALUES (?, ?)",
        (title, json.dumps(vec))
    )
conn.commit()
conn.close()
print("Tabela SQLite została pomyślnie utworzona i zaludniona danymi.")

# 3. Funkcja wyszukiwania semantycznego korzystająca z bazy danych SQLite
def semantic_search_sqlite(query_vec, db_path="movies.db", top_k=3):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT title, embedding FROM Movies")
    rows = cursor.fetchall()
    conn.close()
    
    results = []
    query_norm = np.linalg.norm(query_vec)
    if query_norm == 0:
        return []
        
    for title, emb_str in rows:
        emb_list = json.loads(emb_str)
        doc_vec = np.array(emb_list)
        doc_norm = np.linalg.norm(doc_vec)
        
        if doc_norm == 0:
            sim = 0.0
        else:
            # Podobieństwo cosinusowe
            sim = np.dot(query_vec, doc_vec) / (query_norm * doc_norm)
            
        results.append((title, sim))
        
    # Sortowanie od najbardziej podobnych (malejąco)
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

# Przykładowe zapytanie
query = np.array([0.7, 0.3, 0.8]) # "coś jak sci-fi"
results = semantic_search_sqlite(query, "movies.db", top_k=3)

print("\n--- Wyniki wyszukiwania (SQLite + NumPy) ---")
for title, sim in results:
    print(f"{title}: {sim:.3f}")

Tabela SQLite została pomyślnie utworzona i zaludniona danymi.

--- Wyniki wyszukiwania (SQLite + NumPy) ---
Matrix: 1.000
Incepcja: 0.999
Szeregowiec Ryan: 0.986


## Część 2: Symulacja w czystym Pythonie z NumPy (Zgodnie z szablonem zadania)

In [2]:
# Zadanie 3 BONUS -- symulacja wyszukiwania wektorowego w czystym Pythonie
import numpy as np

# "Baza" filmów z embeddingami (w prawdziwym systemie: OpenAI API)
filmy = {
    "Incepcja":       np.array([0.8, 0.3, 0.9]),
    "Matrix":         np.array([0.75, 0.35, 0.85]),
    "Toy Story":      np.array([0.2, 0.9, 0.1]),
    "Shrek":          np.array([0.25, 0.85, 0.15]),
    "Szeregowiec Ryan": np.array([0.6, 0.1, 0.7]),
}

# Funkcja semantic_search(query_vec, database, top_k=3)
def semantic_search(query_vec, database, top_k=3):
    results = []
    query_norm = np.linalg.norm(query_vec)
    if query_norm == 0:
        return []
        
    for title, doc_vec in database.items():
        doc_norm = np.linalg.norm(doc_vec)
        if doc_norm == 0:
            sim = 0.0
        else:
            # Podobieństwo cosinusowe
            sim = np.dot(query_vec, doc_vec) / (query_norm * doc_norm)
        results.append((title, sim))
        
    # Sortowanie malejąco po podobieństwie
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

# Testowanie wyszukiwania
query = np.array([0.7, 0.3, 0.8]) # "cos jak sci-fi"
results = semantic_search(query, filmy, top_k=3)

print("--- Wyniki wyszukiwania (czysty Python + NumPy) ---")
for title, sim in results:
    print(f"{title}: {sim:.3f}")

--- Wyniki wyszukiwania (czysty Python + NumPy) ---
Matrix: 1.000
Incepcja: 0.999
Szeregowiec Ryan: 0.986
